In [9]:


import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from sklearn.linear_model import (
    LinearRegression,
    Ridge,
    Lasso,
    ElasticNet
)

from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor

from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    AdaBoostRegressor
)

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

In [10]:
df = pd.read_csv('../data/processed/preprocessed.csv')

In [11]:
X = df.drop("price", axis=1)
y = df["price"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [16]:
# Pipeline of ML Models

scaled_models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(),
    "Lasso Regression": Lasso(),
    "ElasticNet Regression": ElasticNet(),
    "KNN Regressor": KNeighborsRegressor(),
    "Support Vector Regressor": SVR(C=1000)
}

tree_models = {
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(random_state=42),
    "Extra Trees": ExtraTreesRegressor(random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
    "AdaBoost": AdaBoostRegressor(random_state=42),
    "XGBoost": XGBRegressor(random_state=42),
    "LightGBM": LGBMRegressor(random_state=42),
    "CatBoost": CatBoostRegressor(random_state=42, verbose=0)
}

trained_models = {}
results = []

for name, model in {**scaled_models, **tree_models}.items():

    # Scaled models
    if name in scaled_models:
        pipeline = Pipeline([
            ("scaler", StandardScaler()),
            ("model", model)
        ])

    # Tree-based models
    else:
        pipeline = Pipeline([
            ("model", model)
        ])

    pipeline.fit(X_train, y_train)

    trained_models[name] = pipeline

    # Predict
    y_pred = pipeline.predict(X_test)

    # Evaluate
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    results.append({
        "Model": name,
        "R2 Score": r2,
        "MAE": mae,
        "RMSE": rmse
    })

# Convert to DataFrame
results_df = pd.DataFrame(results)

# Sort by highest R2 Score
results_df = results_df.sort_values(
    by="R2 Score",
    ascending=False
).reset_index(drop=True)

print("--- Final Model Performance ---")
print(results_df)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000566 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1198
[LightGBM] [Info] Number of data points in the train set: 17290, number of used features: 10
[LightGBM] [Info] Start training from score 537768.047947
--- Final Model Performance ---
                       Model  R2 Score            MAE           RMSE
0                   CatBoost  0.863996   73550.423184  143389.990223
1                   LightGBM  0.842174   77608.524451  154465.537543
2                    XGBoost  0.824679   78382.557172  162801.795755
3              Random Forest  0.814276   77955.868555  167562.537651
4          Gradient Boosting  0.806968   87522.606530  170827.307676
5                Extra Trees  0.804454   80345.151470  171935.925188
6              KNN Regressor  0.757923   98010.756095  191301.811238
7              Decision Tree  0.687444  106252.652903  217373.361605


In [19]:
top_model = results_df.iloc[0]

print(f"\nBest Model: {top_model['Model']}")
print(f"R2 Score: {top_model['R2 Score']:.4f}")
print(f"MAE: {top_model['MAE']:.2f}")
print(f"RMSE: {top_model['RMSE']:.2f}")


Best Model: CatBoost
R2 Score: 0.8640
MAE: 73550.42
RMSE: 143389.99
